# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadfarhan2157-source/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Lane 2 is ranking/scoring, so I want a model that outputs a probability/score, not a hard class — evaluated with Precision@50 to match how the queue is actually used (a reviewer works down a ranked list). Per the lane guide's method menu, I'm training Logistic Regression (interpretable baseline-of-models) and Random Forest (the starter pipeline already showed random forest beating hand rules by ~3x on Precision 50), then comparing both against my Week-4 hand-written rule score on the same split and metric — not against each other in isolation.

In [4]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql(f"""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

path_march = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
path_april = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

# quick test — this should actually hit the dataset
test = con.sql(f"SELECT * FROM '{path_march}' LIMIT 3").df()
print("Connected and read succeeded.")
test

Connected and read succeeded.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
candidate_methods = ["Logistic Regression", "Decision Tree", "Random Forest", "Gradient Boosting"]
chosen = ["Logistic Regression", "Random Forest"]
print("Chosen:", chosen, "— evaluated on Precision@50, compared against Week-4 baseline rule")

Chosen: ['Logistic Regression', 'Random Forest'] — evaluated on Precision@50, compared against Week-4 baseline rule


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Client-grouped, time-aware split. Features come from March 2026, the label comes from April 2026 (strictly future, no overlap with features) — matching the leak lesson from w03. Split is grouped by client_hash_id so no client's pages appear in both train and test; without this, the model could memorize client-level baseline behavior instead of learning transferable signal, which is exactly the risk the lane guide flags for this data.

In [7]:
schema = con.sql(f"DESCRIBE SELECT * FROM '{path_march}'").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [8]:
path_march = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
path_april = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

from sklearn.model_selection import GroupShuffleSplit

march = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_march,
           SUM(gsc_clicks) AS clicks_march,
           AVG(gsc_avg_position) AS avg_position_march,
           SUM(gsc_clicks)/NULLIF(SUM(gsc_impressions),0) AS ctr_march
    FROM '{path_march}'
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
""").df()

april = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS clicks_april
    FROM '{path_april}'
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
""").df()

data = march.merge(april, on=["content_hash_id", "client_hash_id"])
data["declined"] = (data["clicks_april"] < data["clicks_march"]).astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
train_idx, test_idx = next(gss.split(data, groups=data["client_hash_id"]))
train, test = data.iloc[train_idx], data.iloc[test_idx]

overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print("Train rows:", len(train), "Test rows:", len(test), "Client overlap (should be 0):", len(overlap))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train rows: 125834 Test rows: 32715 Client overlap (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

feature_cols = ["impressions_march", "avg_position_march", "ctr_march"]

X_train, y_train = train[feature_cols], train["declined"]
X_test, y_test = test[feature_cols], test["declined"]

# Week-4 baseline rule, applied as a score on this same test set
# (stale_flag dropped for now — needs content_age_days from dim_content, not yet joined)
test = test.copy()
test["ctr_gap_flag"] = ((test["avg_position_march"] > 0) & (test["avg_position_march"] <= 20) &
                         (test["ctr_march"] < 0.02) & (test["impressions_march"] >= 500)).astype(int)
baseline_score = test["ctr_gap_flag"].astype(float)

def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).nlargest(k).index
    return y_true.iloc[top_k_idx].mean()

logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, random_state=1).fit(X_train, y_train)

logreg_scores = logreg.predict_proba(X_test)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "method": ["Week-4 baseline rule (CTR-gap only)", "Logistic Regression", "Random Forest"],
    "roc_auc": [
        roc_auc_score(y_test, baseline_score),
        roc_auc_score(y_test, logreg_scores),
        roc_auc_score(y_test, rf_scores),
    ],
    "precision_at_50": [
        precision_at_k(y_test.reset_index(drop=True), baseline_score.reset_index(drop=True)),
        precision_at_k(y_test.reset_index(drop=True), pd.Series(logreg_scores)),
        precision_at_k(y_test.reset_index(drop=True), pd.Series(rf_scores)),
    ],
})
results

,method,roc_auc,precision_at_50
0,Week-4 baseline rule (CTR-gap only),0.736862,0.42
1,Logistic Regression,0.832923,0.82
2,Random Forest,0.888936,0.90


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)

test_eval = test.copy()
test_eval["rf_score"] = rf_scores
test_eval["actual"] = y_test.values
false_positives = test_eval[(test_eval["rf_score"] > 0.7) & (test_eval["actual"] == 0)]
false_negatives = test_eval[(test_eval["rf_score"] < 0.3) & (test_eval["actual"] == 1)]
print("False positives (high score, didn't decline):", len(false_positives))
print("False negatives (low score, did decline):", len(false_negatives))


Random Forest feature importances:
ctr_march             0.555385
impressions_march     0.248459
avg_position_march    0.196155
dtype: float64
False positives (high score, didn't decline): 2854
False negatives (low score, did decline): 266


In [12]:
base_rate = data["declined"].mean()
print(f"Base rate (share of pages that declined): {base_rate:.3f}")

Base rate (share of pages that declined): 0.279


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.